- Feature Engineering and Preprocessing for F1 Race Prediction
============================================================
- Ce script prépare les données pour l'entraînement des modèles ML

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
from pathlib import Path


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score

In [1]:
def preprocess_data():
    """
    Préparer les données F1 pour l'entraînement
    """
    
    print("🏎️ PREPROCESSING F1 - VERSION SIMPLE")
    print("=" * 50)
    
    # 1. CHARGER LES DONNÉES
    print("\nÉtape 1/4 : Chargement des données...")
    df = pd.read_excel("../data/raw/f1_cleaned.xlsx")
    print(f"{len(df)} courses chargées")
    
    
    # 2. GARDER SEULEMENT LES COLONNES IMPORTANTES
    print("\nÉtape 2/4 : Sélection des colonnes...")
    
    colonnes_importantes = [
        'Season',           # Année
        'Round',            # Numéro de course dans la saison
        'Grid',             # Position de départ
        'Rain',             # Pluie ou pas (0/1)
        'DriverID',         # ID du pilote
        'ConstructorName',  # Nom de l'écurie
        'CircuitID',        # ID du circuit
        'Winner',           # A gagné ? (0/1)
        'Podium',           # Top 3 ? (Yes/No)
        'Position'          # Position finale
    ]
    
    df = df[colonnes_importantes].copy()
    print(f"{len(colonnes_importantes)} colonnes gardées")
    
    
    # 3. ENCODER LES TEXTES EN NOMBRES
    print("\nÉtape 3/4 : Conversion texte → nombres...")
    
    # Convertir les catégories en nombres
    df['DriverID_code'] = df['DriverID'].astype('category').cat.codes
    df['ConstructorName_code'] = df['ConstructorName'].astype('category').cat.codes
    df['CircuitID_code'] = df['CircuitID'].astype('category').cat.codes
    
    # Convertir Podium en 0/1
    df['Podium_num'] = (df['Podium'] == 'Yes').astype(int)
    
    print("Textes convertis en nombres")
    
    
    # 4. SAUVEGARDER
    print("\nÉtape 4/4 : Sauvegarde...")
    
    # Créer le dossier si n'existe pas
    Path("../data/processed").mkdir(parents=True, exist_ok=True)
    
    # Sauvegarder
    df.to_csv("../data/processed/f1_simple.csv", index=False)
    print("Fichier sauvegardé: data/processed/f1_simple.csv")
    
    
    print("\n" + "=" * 50)
    print("PREPROCESSING TERMINÉ !")
    print("=" * 50)
    
    return df


In [3]:
def train_model_simple():
    """
    Entraîner un modèle de prédiction F1
    """
    
    print("🤖 ENTRAÎNEMENT F1 - VERSION SIMPLE")
    print("=" * 50)
    
    
    # 1. CHARGER LES DONNÉES
    print("\n📂 Étape 1/5 : Chargement des données...")
    df = pd.read_csv("../data/processed/f1_simple.csv")
    print(f"   ✅ {len(df)} courses chargées")
    
    
    # 2. PRÉPARER X (features) et y (target)
    print("\n🎯 Étape 2/5 : Préparation X et y...")
    
    # Features simples (colonnes pour prédire)
    X = df[[
        'Grid',                    # Position départ
        'Rain',                    # Pluie
        'Round',                   # Numéro course
        'Season',                  # Année
        'DriverID_code',           # Pilote
        'ConstructorName_code',    # Écurie
        'CircuitID_code'           # Circuit
    ]]
    
    # Target (ce qu'on veut prédire)
    y = df['Winner']  # 0 = pas gagné, 1 = gagné
    
    print(f"   ✅ X: {X.shape[1]} colonnes")
    print(f"   ✅ y: {y.sum()} victoires sur {len(y)} courses")
    
    
    # 3. SÉPARER TRAIN / TEST (80% / 20%)
    print("\n✂️ Étape 3/5 : Séparation train/test...")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.2,      # 20% pour tester
        random_state=42     # Pour avoir toujours le même résultat
    )
    
    print(f"   ✅ Train: {len(X_train)} courses")
    print(f"   ✅ Test:  {len(X_test)} courses")
    
    
    # 4. ENTRAÎNER LES MODÈLES
    print("\n🚀 Étape 4/5 : Entraînement des modèles...")
    
    # --- MODÈLE 1 : RANDOM FOREST ---
    print("\n   Modèle 1: Random Forest...")
    rf_model = RandomForestClassifier(
        n_estimators=100,    # 100 arbres
        max_depth=10,        # Profondeur max
        random_state=42
    )
    rf_model.fit(X_train, y_train)
    
    # Tester
    rf_predictions = rf_model.predict(X_test)
    rf_accuracy = accuracy_score(y_test, rf_predictions)
    print(f"   ✅ Précision: {rf_accuracy:.2%}")
    
    
    # --- MODÈLE 2 : XGBOOST ---
    print("\n   Modèle 2: XGBoost...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42
    )
    xgb_model.fit(X_train, y_train)
    
    # Tester
    xgb_predictions = xgb_model.predict(X_test)
    xgb_accuracy = accuracy_score(y_test, xgb_predictions)
    print(f"   ✅ Précision: {xgb_accuracy:.2%}")
    
    
    # 5. SAUVEGARDER LE MEILLEUR MODÈLE
    print("\n💾 Étape 5/5 : Sauvegarde du meilleur modèle...")
    
    # Choisir le meilleur
    if xgb_accuracy > rf_accuracy:
        best_model = xgb_model
        best_name = "XGBoost"
        best_accuracy = xgb_accuracy
    else:
        best_model = rf_model
        best_name = "Random Forest"
        best_accuracy = rf_accuracy
    
    # Créer dossier
    Path("../models").mkdir(parents=True, exist_ok=True)
    
    # Sauvegarder
    joblib.dump(best_model, "../models/f1_model_simple.joblib")
    
    print(f"   🏆 Meilleur: {best_name}")
    print(f"   ✅ Précision: {best_accuracy:.2%}")
    print(f"   ✅ Sauvegardé: models/f1_model_simple.joblib")
    
    
    print("\n" + "=" * 50)
    print("✅ ENTRAÎNEMENT TERMINÉ !")
    print("=" * 50)
    
    return best_model, best_accuracy
